# 14 — Intelligence Evidence Quickstart

Evidence collection pipeline (SEC EDGAR, Polygon News, DeGiro News). Does NOT call the LLM. See the [intelligence README](https://github.com/matteolongo/swing_screener/blob/main/src/swing_screener/intelligence/README.md) for full documentation.

In [1]:
from __future__ import annotations
import json
import os
from datetime import date
from pathlib import Path

# Ensure CWD is the project root so cache-path resolution works.
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "pyproject.toml").exists():
        os.chdir(p)
        break

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

from swing_screener.intelligence.evidence.collect import collect_evidence
from swing_screener.intelligence.config_access import effective_intelligence_config
from swing_screener.intelligence.evidence.models import SourceEvidence

## Effective Intelligence Config

Resolve the runtime intelligence contract. Evidence settings are nested under the `evidence` key.

In [2]:
cfg = effective_intelligence_config()
evidence_cfg = cfg.get("evidence", {})
print(f"Enabled evidence sources: {evidence_cfg.get('enabled_sources', [])}")
print(f"Recency window days: {evidence_cfg.get('recency_window_days', 'N/A')}")
print(f"Max items per symbol: {evidence_cfg.get('max_items_per_symbol', 'N/A')}")
print(f"Evidence config keys: {list(evidence_cfg.keys())}")

Enabled evidence sources: ['sec_edgar_catalysts', 'polygon_news', 'degiro_news', 'tavily_news']
Recency window days: 3
Max items per symbol: 8
Evidence config keys: ['enabled_sources', 'recency_window_days', 'max_items_per_symbol', 'sec_forms', 'http']


## Collect Evidence for AAPL

`collect_evidence` runs all enabled collectors and returns `list[SourceEvidence]`. Each item has:
- `title` — headline or filing description
- `url` — link to the filing or article
- `publisher` — source name (e.g. `SEC EDGAR`)
- `published_at` — date string
- `quote_or_summary` — excerpt or summary
- `relevance` — relevance label

In [3]:
evidence = collect_evidence("AAPL")
print(f"Items collected: {len(evidence)}")
for item in evidence[:3]:
    print(f"  - [{item.publisher or 'unknown'}] {item.title}")
    print(f"    {item.quote_or_summary[:100] if item.quote_or_summary else '(no summary)'}")

Failed to write evidence cache data/intelligence/evidence/2026-07-28/AAPL.json
Traceback (most recent call last):
  File "/home/memphis/projects/swing_screener/src/swing_screener/intelligence/evidence/collect.py", line 55, in _write_cache
    path.write_text(json.dumps([item.model_dump() for item in items]))
  File "/home/memphis/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/pathlib.py", line 1078, in write_text
    with self.open(mode='w', encoding=encoding, errors=errors, newline=newline) as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/memphis/.local/share/uv/python/cpython-3.11.15-linux-x86_64-gnu/lib/python3.11/pathlib.py", line 1044, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
PermissionError: [Errno 13] Permission denied: 'data/intelligence/evidence/2026-07-28/AAPL.json'


Items collected: 8
  - [REFINITIV_LATEST_NEWS] Apple Inc  <Origin Href="QuoteRef">AAPL.OQ</Origin>  expected to post earnings of $1.89 a share - Earnings Preview
    * Apple Inc <AAPL.OQ> <AAPL.O> is expected to  show a rise
in quarterly revenue when it reports  res
  - [REFINITIV_LATEST_NEWS] Apple Hospitality REIT stockt Kreditfazilität auf 1,3 Milliarden US-Dollar auf und verlängert Laufzeiten
    *      Apple Hospitality REIT refinanziert unbesicherte Kreditlinien;
Gesamtkapazität der Main Credi
  - [REFINITIV_LATEST_NEWS] MERCATI IN TEMPO REALE-Wall Street chiude con andamenti contrastanti: la Fed e i titoli delle megacap al centro dell&apos;attenzione
    ((Traduzione automatizzata da Reuters usando Machine Learning e IA
generativa. Si prega di consultar


## Cached Evidence Summary

Evidence is cached to `data/intelligence/evidence/{date}/{TICKER}.json`. The helper below reads the most recent cache for a ticker.

In [4]:
def read_latest_cached_evidence_summary(ticker: str) -> str | None:
    """Read the latest cached evidence for *ticker* and return a summary string."""
    cache_root = Path("data/intelligence/evidence")
    if not cache_root.exists():
        return None
    dates = sorted(
        d for d in cache_root.iterdir() if d.is_dir() and d.name.replace("-", "").isdigit()
    )
    if not dates:
        return None
    cache_file = dates[-1] / f"{ticker.upper()}.json"
    if not cache_file.exists():
        return None
    try:
        raw = json.loads(cache_file.read_text())
        items = [SourceEvidence(**d) for d in raw]
        return f"{len(items)} item(s), latest: {items[0].title if items else 'empty'}"
    except (OSError, ValueError, TypeError):
        return None


summary = read_latest_cached_evidence_summary("AAPL")
print(f"Cached: {summary}")

Cached: None
